In [ ]:
import numpy as np
import torch
import torch.nn as nn
import pandas as pd
from sklearn.utils import resample
import tune
from torch.profiler import profile, record_function, ProfilerActivity
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import roc_auc_score

In [101]:
pipe, x, y = tune.process_data()

In [3]:
# from sklearn.model_selection import train_test_split

# X_train, X_test, y_train, y_test = \
#     train_test_split(x, y, 
#                     test_size=0.20,
#                     stratify=y,
#                     random_state=1)

# X_train = pipe.fit_transform(X_train, y_train)
# X_test = pipe.transform(X_test)
# y_train = y_train.to_numpy()
# y_train = y_train.reshape(-1, 1)
# y_test = y_test.to_numpy()
# y_test = y_test.reshape(-1, 1)

In [34]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [5]:
# from torch.utils.data import DataLoader, TensorDataset

# X_train = torch.tensor(X_train).float()
# y_train = torch.tensor(y_train).float()
# X_train = X_train.to(device)
# y_train = y_train.to(device)
# train_ds = TensorDataset(X_train, y_train)

# batch_size = 256
# torch.manual_seed(1)
# train_dl = DataLoader(train_ds, batch_size, shuffle=True)

# X_test = torch.tensor(X_test).float()
# y_test = torch.tensor(y_test).float()
# X_test = X_test.to(device)
# y_test = y_test.to(device)
# test_dl = DataLoader(TensorDataset(X_test, y_test), batch_size=batch_size, shuffle=False)

In [ ]:
# input_size = X_train.shape[1]
# hidden_units = [input_size, input_size]

# all_layers = nn.ModuleList()
# for hidden_unit in hidden_units:
#     layer = nn.Linear(input_size, hidden_unit)
#     all_layers.append(layer)
#     all_layers.append(nn.ReLU())
# all_layers.append(nn.Linear(hidden_units[-1], 1)) 
# all_layers.append(nn.Sigmoid())  
# model = nn.Sequential(*all_layers)
# model.to(device)

In [103]:
class PitNN(nn.Module):
    
    def __init__(self):
        super(PitNN, self).__init__()
        self.l1 = nn.Linear(14, 28)
        self.l2 = nn.Linear(28, 14)
        self.relu = nn.ReLU()
        self.lf = nn.Linear(14, 1)
        self.sigmoid = nn.Sigmoid()
    def forward(self, x):
        x = self.relu(self.l1(x))
        # x = self.relu(self.l1(x))
        x = self.relu(self.l2(x))
        # x = self.relu(self.l1(x))
        # x = self.relu(self.l1(x))
        # x = self.relu(self.l1(x))
        # x = self.relu(self.l1(x))
        x = self.sigmoid(self.lf(x))
        return x

In [36]:
def evaluate(model, dataloader, device):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for x_batch, y_batch in dataloader:
            pred = model(x_batch.to(device))
            is_correct = ((pred > 0.5).float() == y_batch.to(device)).float()
            correct += is_correct.sum().cpu()
            total += y_batch.size(0)
    accuracy = correct / total
    return accuracy

In [69]:
def train_model(model, X_train, y_train, X_test=None, y_test=None, batch_size=256, device='gpu'):
    
    loss_fn = nn.BCELoss()  
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=0.0001)
    torch.manual_seed(1)
    num_epochs = 20
    train_ds = TensorDataset(torch.tensor(X_train).float(), torch.tensor(y_train).float())
    train_dl = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    if X_test is not None and y_test is not None:
        test_ds = TensorDataset(torch.tensor(X_test).float(), torch.tensor(y_test).float())
        test_dl = DataLoader(test_ds, batch_size=batch_size, shuffle=False)
    else:
        test_dl = None

    for epoch in range(num_epochs):
        model.train()
        accuracy_hist_train = 0
        training_history = []
        validation_history = []
        for x_batch, y_batch in train_dl:
            pred = model(x_batch.to(device))
            loss = loss_fn(pred, y_batch.to(device))
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
            is_correct = ((pred > 0.5).float() == y_batch.to(device)).float()
            accuracy_hist_train += is_correct.sum().cpu()
        accuracy_hist_train /= len(train_dl.dataset)
        if test_dl is not None:
            validation_accuracy = evaluate(model, test_dl, device)
            validation_history.append(validation_accuracy)
        training_history.append(accuracy_hist_train)
        print(f'Epoch {epoch}  Accuracy {accuracy_hist_train:.4f}  Validation Accuracy {validation_accuracy:.4f}' if test_dl is not None else f'Epoch {epoch}  Accuracy {accuracy_hist_train:.4f}')
    return validation_accuracy if test_dl is not None else accuracy_hist_train

In [73]:
#class_weights = torch.tensor([0.9]).to(device)

from sklearn.model_selection import StratifiedKFold

kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=1)
cv_scores = []
for train, test in kfold.split(x, y):
    model = PitNN().to(device)
    X_train = pipe.fit_transform(x.iloc[train], y.iloc[train])
    X_test = pipe.transform(x.iloc[test])
    y_train = y.iloc[train].to_numpy().reshape(-1, 1)
    y_test = y.iloc[test].to_numpy().reshape(-1, 1)
    score = train_model(model, X_train, y_train, X_test, y_test, batch_size=256, device=device)
    cv_scores.append(score)
print(f'Average CV Score: {np.mean(cv_scores)}')

Epoch 0  Accuracy 0.8519  Validation Accuracy 0.8755
Epoch 1  Accuracy 0.8767  Validation Accuracy 0.8763
Epoch 2  Accuracy 0.8778  Validation Accuracy 0.8793
Epoch 3  Accuracy 0.8791  Validation Accuracy 0.8782
Epoch 4  Accuracy 0.8804  Validation Accuracy 0.8804
Epoch 5  Accuracy 0.8815  Validation Accuracy 0.8815
Epoch 6  Accuracy 0.8824  Validation Accuracy 0.8812
Epoch 7  Accuracy 0.8822  Validation Accuracy 0.8824
Epoch 8  Accuracy 0.8825  Validation Accuracy 0.8823
Epoch 9  Accuracy 0.8829  Validation Accuracy 0.8825
Epoch 10  Accuracy 0.8829  Validation Accuracy 0.8824
Epoch 11  Accuracy 0.8830  Validation Accuracy 0.8826
Epoch 12  Accuracy 0.8833  Validation Accuracy 0.8830
Epoch 13  Accuracy 0.8835  Validation Accuracy 0.8831
Epoch 14  Accuracy 0.8833  Validation Accuracy 0.8836
Epoch 15  Accuracy 0.8835  Validation Accuracy 0.8840
Epoch 16  Accuracy 0.8837  Validation Accuracy 0.8821
Epoch 17  Accuracy 0.8834  Validation Accuracy 0.8831
Epoch 18  Accuracy 0.8838  Validation 

In [68]:
model = PitNN().to(device)
X_train = pipe.fit_transform(x, y)
y_train = y.to_numpy().reshape(-1, 1)
train_model(model, X_train, y_train, batch_size=256, device=device)

Epoch 0  Accuracy 0.8274
Epoch 1  Accuracy 0.8733
Epoch 2  Accuracy 0.8761
Epoch 3  Accuracy 0.8773
Epoch 4  Accuracy 0.8782
Epoch 5  Accuracy 0.8790
Epoch 6  Accuracy 0.8793
Epoch 7  Accuracy 0.8795
Epoch 8  Accuracy 0.8801
Epoch 9  Accuracy 0.8806
Epoch 10  Accuracy 0.8809
Epoch 11  Accuracy 0.8809
Epoch 12  Accuracy 0.8815
Epoch 13  Accuracy 0.8812
Epoch 14  Accuracy 0.8815
Epoch 15  Accuracy 0.8816
Epoch 16  Accuracy 0.8815
Epoch 17  Accuracy 0.8813
Epoch 18  Accuracy 0.8819
Epoch 19  Accuracy 0.8818
Epoch 20  Accuracy 0.8816
Epoch 21  Accuracy 0.8817
Epoch 22  Accuracy 0.8818
Epoch 23  Accuracy 0.8818
Epoch 24  Accuracy 0.8818
Epoch 25  Accuracy 0.8821
Epoch 26  Accuracy 0.8817
Epoch 27  Accuracy 0.8820
Epoch 28  Accuracy 0.8818
Epoch 29  Accuracy 0.8817
Epoch 30  Accuracy 0.8818
Epoch 31  Accuracy 0.8817
Epoch 32  Accuracy 0.8818
Epoch 33  Accuracy 0.8817
Epoch 34  Accuracy 0.8817
Epoch 35  Accuracy 0.8821
Epoch 36  Accuracy 0.8819
Epoch 37  Accuracy 0.8817
Epoch 38  Accuracy 0.8

KeyboardInterrupt: 

In [ ]:
model.eval()
model.to('cpu')
X_test_t = X_test.to('cpu')
y_test = y_test.to('cpu')
with torch.no_grad():
    y_pred = model(X_test_t)
y_pred = (y_pred > 0.5).float().numpy()
y_test_np = y_test.numpy().astype(int)

from sklearn.metrics import classification_report
print(classification_report(y_test_np, y_pred))

In [ ]:
from sklearn.metrics import roc_auc_score
auc_score = roc_auc_score(y_test, y_pred)
print(f'ROC AUC Score: {auc_score:.4f}')

In [ ]:
# testing_data = pd.read_csv("./data/test.csv")
# preds = model(torch.tensor(pipe.transform(testing_data.drop(columns=["id"]))).float().to(device)).cpu().detach().numpy()
# submission = pd.DataFrame({"id": testing_data["id"], "target": preds.flatten()})
# submission.to_csv("./data/nn_baseline_submission.csv", index=False)

In [ ]:
# from skorch import NeuralNetBinaryClassifier, NeuralNetClassifier
# pipe, x, y = tune.process_data()
# optimizer = torch.optim.Adam(PitNN().parameters(), lr=0.001, weight_decay=0.0001)
# net = NeuralNetBinaryClassifier(
#     PitNN,
#     # Shuffle training data on each epoch
#     iterator_train__shuffle=True,
#     # Don't use any callbacks
#     callbacks=[],
#     # Use the GPU if available
#     device=device,
#     # Train the model for 20 epochs
#     max_epochs=20,
#     # Use binary cross-entropy loss
#     criterion=nn.BCELoss,
#     # Use the Adam optimizer with a learning rate of 0.001
#     optimizer=torch.optim.Adam,
#     optimizer__lr=0.001,
#     optimizer__weight_decay=0.0001,

#     # Use a batch size of 256
#     batch_size=256,
# )

# # X_skorch = pipe.fit_transform(x, y).astype(np.float32)
# # y_skorch = y.to_numpy().reshape(-1, 1).astype(np.float32)
# pipe.steps.append(['classifier_nn', net])
# print(x.dtypes)
# pipe.fit(x, y)

Driver                     object
Compound                   object
Race                       object
Year                        int64
PitStop                     int64
LapNumber                   int64
Stint                       int64
TyreLife                  float64
Position                    int64
LapTime (s)               float64
LapTime_Delta             float64
Cumulative_Degradation    float64
RaceProgress              float64
Position_Change           float64
dtype: object


RuntimeError: mat1 and mat2 must have the same dtype, but got Double and Float